In [19]:

import sys
import os
import importlib

# Remove all lingo_to_pyomo modules from cache to force reload
modules_to_remove = [m for m in sys.modules if 'lingo_to_pyomo' in m or 'json_parser' in m or 'lingo' in m or 'notebook_generator' in m or 'excel_parser' in m]
for m in modules_to_remove:
    del sys.modules[m]

print(f"Cleared {len(modules_to_remove)} cached modules")

# Now re-import
src_path = os.path.abspath("../src")
sys.path.insert(0, src_path)

from lingo_parser.parser import *

from lingo_parser.transformer import *
from pyomo_generator.json_parser import *
from notebook_generator.notebook_construct import *
from excel_parser.excel_module import *

print("Reloaded all modules")


Cleared 8 cached modules
Reloaded all modules


In [ ]:


def generate_notebook(input_file, output_notebook, external_data=False) :

    ext_d = external_data
    tree = parse_lingo_model("../data/Cardoza.lng")
    model_dict = LingoModelTransformer2().transform(tree)

    pyomo_code = generate_pyomo_code(model_dict, external_data=ext_d)
    save_pyomo_data_to_json(model_dict)


    generate_pyomo_notebook(pyomo_code, solver="highs", filename=f"{output_notebook}.ipynb",json_data_filename="./data/pyomo_data.json",external_data=ext_d)

generate_notebook()

✅ Notebook généré : Cardozatest.ipynb


In [58]:
file = "../data/Delivery.lng"

ext_d = False
convert_lingo_ole_to_explicit(file)

'/Users/joaquim/Documents/CODE/Lingpy/lingo_to_pyomo/data/Delivery_explicit.lng'

In [59]:
connexion = [1,0,0,0,1,0,0,0,1,0,0,1,0,1,0,1,0,0,1,1,0,0,1,1,0,0,1,0,1,0,1,0,0,0,0,1,0,1,0,0,0,0,1,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,1,0,0,1,0,1,0,0,0,0,1,0,1,0,1,0,0,1,0,0,0]
len(connexion)


90

In [20]:

ext_d = False

tree = parse_lingo_model("../data/Pastissimo.lng")
model_dict = LingoModelTransformer2().transform(tree)

pyomo_code = generate_pyomo_code(model_dict, external_data=ext_d)
print(pyomo_code)

from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.PERIODES = Set(initialize=[1, 2, 3, 4, 5, 6])

#==============================================================================
# PARAMETERS
#==============================================================================

model.Cout_achat = Param(model.PERIODES, initialize={1: 1000.0, 2: 975.0, 3: 1000.0, 4: 980.0, 5: 1020.0, 6: 1025.0}, within=NonNegativeReals)
model.Cout_prod = Param(model.PERIODES, initialize={1: 160.0, 2: 150.0, 3: 150.0, 4: 160.0, 5: 175.0, 6: 165.0}, within=NonNegativeReals)
model.mini = Param(model.PERIODES, initialize={1: 4.0, 2: 3.0, 3: 5.0, 4: 2.0, 5: 4.0, 6: 5.0}, within=NonNegativeReals)
model.maxi = Param(model.PERIODES, initialize={1: 6.0, 2: 4.0, 3: 7.0, 4: 3.0, 5: 7.0, 6: 6.0}, within=NonNegativeReals)
model.cap_prod = Param(mode

In [17]:


tree = parse_lingo_model("../data/Buckly.lng")
model_dict = LingoModelTransformer2().transform(tree)

save_pyomo_data_to_json(model_dict)



pyomo_code = generate_pyomo_code(model_dict, external_data=False)

#print(pyomo_code)
local_vars = {}
exec(pyomo_code, local_vars)
model = local_vars["model"]

# Résout le modèle avec un solveur (par défaut glpk ou cbc)
solver = SolverFactory("highs")  # ou 'cbc' si glpk n'est pas disponible
result = solver.solve(model, tee=False)


print(model.obj())
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')

1504.0


,Variable,Index,Valeur
0,Xplein,1,3.0000
1,Xplein,2,3.0000
2,Xplein,3,4.0000
3,Xpartiel,1,1.0000
4,Xpartiel,2,2.0000
5,Xpartiel,3,3.0000
6,Xpartiel,4,2.0000


In [64]:
print(pyomo_code)

from pyomo_generator.json_parser import load_pyomo_data
data = load_pyomo_data('./data/pyomo_data.json')

from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.ALIMENTS = Set(initialize=data['sets']['ALIMENTS'])
model.INGREDIENTS = Set(initialize=data['sets']['INGREDIENTS'])
model.ARCS = Set(dimen=2, initialize=[(i,j) for i in model.ALIMENTS for j in model.INGREDIENTS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.Prix = Param(model.ALIMENTS, initialize=data['params']['Prix'], within=NonNegativeReals)
model.Calories = Param(model.ALIMENTS, initialize=data['params']['Calories'], within=NonNegativeReals)
model.DIETEJOUR = Param(model.INGREDIENTS, initialize=data['params']['DIETEJOUR

In [106]:
from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.employes = Set(initialize=[1, 2, 3, 4, 5])
model.taches = Set(initialize=[1, 2, 3, 4, 5])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.employes for j in model.taches])

#==============================================================================
# PARAMETERS
#==============================================================================

model.c = Param(model.employes, model.taches, initialize={(1, 1): 39.0, (1, 2): 65.0, (1, 3): 69.0, (1, 4): 66.0, (1, 5): 57.0, (2, 1): 64.0, (2, 2): 84.0, (2, 3): 24.0, (2, 4): 92.0, (2, 5): 22.0, (3, 1): 49.0, (3, 2): 50.0, (3, 3): 61.0, (3, 4): 31.0, (3, 5): 45.0, (4, 1): 48.0, (4, 2): 45.0, (4, 3): 55.0, (4, 4): 23.0, (4, 5): 50.0, (5, 1): 59.0, (5, 2): 34.0, (5, 3): 30.0, (5, 4): 34.0, (5, 5): 18.0}, within=NonNegativeReals)

#==============================================================================
# VARIABLES
#==============================================================================

model.x = Var(model.employes, model.taches, domain=NonNegativeReals)

#==============================================================================
# CONSTRAINTS
#==============================================================================

model.c0 = Constraint(expr=model.x(5,5) <= 0)
model.c1 = Constraint(expr=model.x(5,2) >= 1)
model.c2 = Constraint(expr=model.x(2,3) <= 0)
model.c3 = Constraint(expr=model.x(2,5) >= 1)
model.c4 = Constraint(expr=model.x(5,3) <= 0)
model.c5 = Constraint(expr=model.x(5,2) >= 1)
model.c_for_0 = ConstraintList()
for t in model.taches:
    model.c_for_0.add(sum(model.x[e,t] for e in model.employes) == 1)
model.c_for_1 = ConstraintList()
for e in model.employes:
    for t in model.taches:
        model.c_for_1.add(model.x(e,t) <= 1)

#==============================================================================
# OBJECTIVE
#==============================================================================

model.obj = Objective(expr=sum(model.c[model.e,model.t] * model.x[model.e,model.t] for model.e,model.t in model.ARC), sense=minimize)

TypeError: 'IndexedVar' object is not callable